# 02 — Cascade data assimilation cycle

This notebook reproduces the **XiChen cycling DA** experiment: starting cold
from climatology, we run three 12 h assimilation windows
(2023-01-05 00 UTC → 2023-01-06 00 UTC), each assimilating six observation
streams — four microwave radiance streams (ATMS, AMSU-A, MHS, HRS4) through
their learned observation operators, plus conventional prepbufr and SATWND —
with the cascade of per-stream 4DVar-gradient-conditioned DA networks.

At every cycle we also compute the latitude-weighted RMSE of background
(**xb**) and analysis (**xa**) against ERA5, so you can watch the analysis
pull the state towards the truth. At 00/12 UTC an additional 6 h window
re-analysis is written to `ic_6h/` — the initial condition used by notebook
**03**.

**Prerequisites**: all six checkpoints in `../ckpts/`, ERA5 + observations
in `../data/` (see README § Download). Runtime: ~5–10 min on one GPU.

In [ ]:
from datetime import datetime
from pathlib import Path

import json
import numpy as np
import torch
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from xichen.data import VARIABLES
from xichen.device import get_device
from xichen.dacycle.common import (
    load_models, load_obs_window, load_xb_initial, load_era5_truth,
    compute_metrics, save_nc, save_metrics, _run_solver,
)
from xichen.dacycle.det import assim_cycle_xichen
from xichen.metrics import weighted_rmse_channels
from xichen.data import get_normalize

# --- paths ---
DATA_DIR = Path("/fs6/home/yangjh_data/project_data/xichen")
ERA5_DIR = Path("/fs6/home/yangjh_data/project_data/xichen/observation/ERA5")
OBS_DIR = Path("/fs6/home/yangjh_data/project_data/xichen/observation")
CKPT_DIR = Path("/fs6/home/yangjh15/xichen/ckpt/xichen_1p0deg")
OUTPUT_DIR = Path("/fs6/home/yangjh15/xichen/XiChen_1p0deg_public/outputs/demo_dacycle")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = get_device("cuda", 0)

# ERA5 1.0° 网格(纬度 90→-90,经度 0→359),供 EqualEarth 投影 pcolormesh
LATS = np.linspace(90.0, -90.0, 181)
LONS = np.linspace(0.0, 359.0, 360)
LON_GRID, LAT_GRID = np.meshgrid(LONS, LATS)

# Mirror of configs/dacycle_demo.json with cluster absolute paths.
cfg = {
    "cycle_start": "2023-01-05T00:00:00",
    "cycle_end": "2023-01-06T00:00:00",
    "cycle_interval_hours": 12,
    "trigger_6h_hours": [0, 12],
    "obs_order": ["atms", "amsua", "mhs", "hrs4", "prepbufr", "satwnd"],
    "era5_lr_dir": str(ERA5_DIR),
    "obs_dir": str(OBS_DIR),
    "scale_dir": str(ERA5_DIR / "normalized_mean_std"),
    "forecast_config": "../configs/xichen_forecast.json",
    "obsop_configs": {s: f"../configs/{s}_obsop.json" for s in ["atms", "amsua", "mhs", "hrs4"]},
    "obs_dict_config": "../configs/obs_dict_6obs.json",
    "ckpt_forecast": str(CKPT_DIR / "xichen_state_forecast_ar15.ckpt"),
    "ckpt_obsop": {s: str(CKPT_DIR / f"xichen_obsop_{s}.ckpt") for s in ["atms", "amsua", "mhs", "hrs4"]},
    "ckpt_cascade_da": str(CKPT_DIR / "xichen_cascade_da_randombg_atms_amsua_mhs_hrs4_prepbufr_satwnd.ckpt"),
    "output_dir": str(OUTPUT_DIR),
    "save_format": "nwp",
    "device": "cuda",
}
print("device:", device)

In [ ]:
# Load every component: forecast + 4 observation operators + 6 DA networks + Solver.
cascade_cfg = json.load(open("../configs/cascade_da_6obs.json"))
obs_dict = json.load(open(cfg["obs_dict_config"]))["obs_dict"]

components = load_models(cfg, cascade_cfg, obs_dict, device)
forecast, obsops, da_models, h_models, varcost_models, solver, microwave_prep, conventional_prep = components
for name, module in [("forecast", forecast), *obsops.items(), *da_models.items()]:
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:>12s}: {n/1e6:6.2f}M params")

In [ ]:
# Three 12 h cycles. For plotting we also keep the background xb of each cycle
# (computed exactly as assim_cycle_xichen does internally: cold start from
# climatology, otherwise the 12 h forecast of the previous analysis).
cycles = [datetime(2023, 1, 5, 0), datetime(2023, 1, 5, 12), datetime(2023, 1, 6, 0)]

history = []
prev_xa = None
for T in cycles:
    # background for plotting (same computation as inside assim_cycle_xichen)
    if prev_xa is None:
        xb_plot = load_xb_initial(cfg, T, device)
    else:
        with torch.no_grad():
            preds, _ = forecast(prev_xa.to(device),
                                torch.tensor([[12 * 0.01]], device=device),
                                VARIABLES, use_checkpoint=True)
            xb_plot = preds.detach().float()

    # production path: 12h window DA (+ 6h re-analysis at 00/12 UTC → ic_6h/)
    xa12, timing = assim_cycle_xichen(cfg, T, prev_xa, components, obs_dict, device)

    truth = load_era5_truth(cfg["era5_lr_dir"], T, cfg["scale_dir"], device)
    history.append({"T": T, "xb": xb_plot.detach().cpu(), "xa": xa12.detach().cpu(),
                    "truth": truth.detach().cpu(), "timing": timing})
    prev_xa = xa12
    print(f"T={T.isoformat()} done in {sum(timing.values()):.1f}s "
          f"(fc {timing['t_forecast_12h_s']:.1f} + da12 {timing['t_da_12h_s']:.1f} "
          f"+ da6 {timing['t_da_6h_s']:.1f})")

In [ ]:
# Analysis increments (xa - xb) in physical units: z-500 and t2m, one row per cycle.
mean_np, std_np = get_normalize(cfg["scale_dir"], VARIABLES)

def phys(x, v):
    """normalized (1,69,H,W) -> physical units of variable v (H,W)."""
    i = VARIABLES.index(v)
    return (x.numpy()[0, i] * std_np[0, i] + mean_np[0, i])

fig, axes = plt.subplots(len(history), 2, figsize=(12, 3.2 * len(history)),
                         constrained_layout=True,
                         subplot_kw={"projection": ccrs.EqualEarth()})
for row, h in enumerate(history):
    for col, v in enumerate(["z-500", "t2m"]):
        inc = phys(h["xa"], v) - phys(h["xb"], v)
        lim = np.nanmax(np.abs(inc))
        pcm = axes[row, col].pcolormesh(LON_GRID, LAT_GRID, inc, cmap="RdBu_r", vmin=-lim, vmax=lim, transform=ccrs.PlateCarree())
        axes[row, col].set_title(f"{v} increment  T={h['T']:%Y-%m-%d %H}UTC")
        axes[row, col].coastlines(linewidth=0.5)
        plt.colorbar(pcm, ax=axes[row, col], shrink=0.85)
fig.savefig(OUTPUT_DIR / "02_analysis_increments.png", dpi=300)
fig.savefig(OUTPUT_DIR / "02_analysis_increments.pdf", dpi=300)
plt.show()

In [ ]:
# Lat-weighted RMSE of xb vs xa against ERA5, per cycle (z-500 / t2m).
labels = [f"{h['T']:%m-%d %H}Z" for h in history]

def rmse_of(state, truth, v):
    a = phys(state, v)[None, None]   # (1, 1, H, W)
    b = phys(truth, v)[None, None]
    return float(weighted_rmse_channels(a, b).squeeze())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, v in zip(axes, ["z-500", "t2m"]):
    rmse_xb = [rmse_of(h["xb"], h["truth"], v) for h in history]
    rmse_xa = [rmse_of(h["xa"], h["truth"], v) for h in history]
    xpos = np.arange(len(labels))
    ax.bar(xpos - 0.2, rmse_xb, width=0.4, label="background xb")
    ax.bar(xpos + 0.2, rmse_xa, width=0.4, label="analysis xa")
    ax.set_xticks(xpos, labels)
    ax.set_title(f"{v} RMSE vs ERA5")
    ax.grid(alpha=0.3, axis="y")
    ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "02_cycle_rmse.png", dpi=300)
fig.savefig(OUTPUT_DIR / "02_cycle_rmse.pdf", dpi=300)
plt.show()

In [ ]:
# Observation coverage of the first 12 h window (ATMS swaths + prepbufr).
obs12, mask12, _ = load_obs_window(
    cfg["obs_dir"], cfg["era5_lr_dir"], history[0]["T"],
    window_hours=12, dt_obs=3, obs_order=cfg["obs_order"], obs_dict=obs_dict,
    microwave_prep=microwave_prep, conventional_prep=conventional_prep,
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), subplot_kw={"projection": ccrs.EqualEarth()})
for ax, obs in zip(axes, ["atms", "prepbufr"]):
    coverage = mask12[obs][0, :, 0].sum(dim=0).numpy()  # n_time slots seen per grid cell
    pcm = ax.pcolormesh(LON_GRID, LAT_GRID, coverage, cmap="viridis", transform=ccrs.PlateCarree())
    ax.set_title(f"{obs} coverage (# of 3h slots with data)")
    ax.coastlines(linewidth=0.5)
    plt.colorbar(pcm, ax=ax, shrink=0.8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "02_obs_coverage.png", dpi=300)
fig.savefig(OUTPUT_DIR / "02_obs_coverage.pdf", dpi=300)
plt.show()

**Outputs** (under `../outputs/demo_dacycle/`):

- `analysis_12h/<T>.nc` — 12 h-window analyses (all three cycles)
- `ic_6h/<T>.nc` — 6 h-window re-analyses at 00/12 UTC (the DA initial condition)
- `analysis_12h_metrics/`, `ic_6h_metrics/` — per-cycle metrics JSON

Next: `03_da_init_forecast.ipynb` — launch a 10-day forecast from the
`ic_6h` analysis and compare it with the ERA5-initialised baseline.